# Phase 7 — Detection/Segmentation Results Analysis

Joins Phase 7 SSD detection results to Phase 3 classification results and tests H1–H4 from
`ideas/PHASE7_PLAN.md`. Replaces `scripts/phase7_analysis.py` (deleted — its run-dir parser was
already broken for the current `ssd_<model>_<stage>[_pretrained]_phase7_detection` naming, and its
H2–H4 were unimplemented stubs).

## Validity banner — read before trusting any number below

The anchor-recall bug (`ideas/PHASE7_PLAN.md` root cause: bad tap indices + a linear-interpolation
scale gap over VOC's actual box-size distribution) was fixed in commit `686b419`
(`docs/PHASE7_LOG.md` Stage 9). **Any run built before that commit is invalid, not just low** —
anchor recall was 0.76–0.80 regardless of resolution, so those models never saw the right ground
truth to learn from. This notebook computes provenance per run (`git merge-base --is-ancestor`)
and refuses to plot pre-fix runs rather than trusting the directory name.

Known pre-fix (superseded) runs as of this writing: `*_fp32_early_30ep`, `*_minratio02`,
`*_diag_256`, `*_diag_512`, and the non-`_pretrained` `fire`/`tv` FP32 runs.

Also true as of this writing:
- No INT8 detection metrics exist — every `*_int8_*` run dir holds only `config.yaml`; the one
  INT8 job that ran (811101) crashed on uncalibrated observers (`min tensor(inf) should be less
  than max tensor(-inf)`). H2 is not testable yet.
- Segmentation (Part B of Phase 7) is unstarted: `build_deeplabv3_segmenter()` is a placeholder,
  `SegmentationTrainer.fit()` is a stub, and the CLI's `run_segmentation()` prints "not yet
  implemented".
- `_pretrained` runs (backbone initialized from the Phase 3 Tiny-ImageNet checkpoint instead of
  random init) are a separate variable from anchor-fix provenance — both are tracked below.

## Configuration

In [1]:
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    """Walk upward until the repository root is found."""
    start = start or Path.cwd()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "results").exists() and (candidate / "models").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))  # so `import ml` resolves regardless of kernel cwd
print(f"Project root: {PROJECT_ROOT}")

PHASE7_DIR = PROJECT_ROOT / "outputs" / "detection_segmentation" / "phase7"
RESULTS_DIR = PROJECT_ROOT / "results" / "phase_7_detection_segmentation_analysis"
FIGURES_DIR = PROJECT_ROOT / "results" / "figures_generated" / "phase_7_detection_segmentation"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# The commit that fixed the anchor-recall bug (tap indices + anchor scales). Runs built
# from a commit that isn't a descendant of this one trained against broken anchor coverage.
ANCHOR_FIX_COMMIT = "686b419"

print(f"Phase 7 run directory: {PHASE7_DIR}")
print(f"Anchor-fix commit: {ANCHOR_FIX_COMMIT}")

Project root: /home/rafael/Documents/dnn_study
Phase 7 run directory: /home/rafael/Documents/dnn_study/outputs/detection_segmentation/phase7
Anchor-fix commit: 686b419


## Load Phase 7 Results

Parses every `ssd_<model>_<stage>[_pretrained]_phase7_detection` run directory directly off disk
— the source of truth is `metrics.json` + `git_hash.txt`, not a hardcoded run list, so this cell
picks up new PCAD results (e.g. the in-flight `fire`/`tv` `_pretrained` jobs) on re-run without
edits.

In [2]:
import re

RUN_NAME_RE = re.compile(
    r"^ssd_(?P<model>alexnet_(?:bottleneck|fire|tv))_(?P<stage>fp32|qat|int8)"
    r"(?P<pretrained>_pretrained)?_(?P<exp>.+)$"
)


def is_post_fix(git_hash: str) -> bool | None:
    """True if `git_hash` is a descendant of ANCHOR_FIX_COMMIT, None if not decidable."""
    if not git_hash:
        return None
    try:
        result = subprocess.run(
            ["git", "merge-base", "--is-ancestor", ANCHOR_FIX_COMMIT, git_hash],
            cwd=PROJECT_ROOT, capture_output=True, timeout=10,
        )
    except (subprocess.SubprocessError, OSError):
        return None
    if result.returncode == 0:
        return True
    if result.returncode == 1:
        return False
    return None  # unknown commit (e.g. squashed/rebased history) — don't assert either way


def run_status(run_dir: Path, has_metrics: bool) -> str:
    if has_metrics:
        return "complete"
    logs = list(run_dir.glob("*.log"))
    return "running" if logs else "not_started"


def load_phase7_runs(phase7_dir: Path = PHASE7_DIR, exp: str = "phase7_detection") -> pd.DataFrame:
    rows = []
    for run_dir in sorted(phase7_dir.glob("ssd_*")):
        if not run_dir.is_dir():
            continue
        m = RUN_NAME_RE.match(run_dir.name)
        if not m or m.group("exp") != exp:
            continue

        git_hash = ""
        git_hash_path = run_dir / "git_hash.txt"
        if git_hash_path.exists():
            git_hash = git_hash_path.read_text().strip()

        metrics_path = run_dir / "metrics.json"
        has_metrics = metrics_path.exists()
        row = {
            "run_dir": run_dir.name,
            "model": m.group("model"),
            "stage": m.group("stage"),
            "pretrained": m.group("pretrained") is not None,
            "git_hash": git_hash,
            "post_fix": is_post_fix(git_hash),
            "status": run_status(run_dir, has_metrics),
            "best_val_mAP": np.nan,
            "best_val_mAP50": np.nan,
            "best_epoch": np.nan,
            "total_time_s": np.nan,
            "params_m": np.nan,
            "macs": np.nan,
            "model_size_mb": np.nan,
            "latency_ms_per_image": np.nan,
        }

        if has_metrics:
            with open(metrics_path) as f:
                metrics = json.load(f)
            row["best_val_mAP"] = metrics.get("best_val_mAP", np.nan)
            row["best_epoch"] = metrics.get("best_epoch", np.nan)
            row["total_time_s"] = metrics.get("total_time_s", np.nan)
            map50 = metrics.get("val_mAP50")
            best_epoch = metrics.get("best_epoch")
            if map50 and best_epoch is not None and 0 <= best_epoch < len(map50):
                row["best_val_mAP50"] = map50[best_epoch]
            summary = metrics.get("summary", {})
            row["params_m"] = summary.get("params_m", np.nan)
            row["macs"] = summary.get("macs", np.nan)
            row["model_size_mb"] = summary.get("model_size_mb", np.nan)
            row["latency_ms_per_image"] = summary.get("latency_ms_per_image", np.nan)

        rows.append(row)

    return pd.DataFrame(rows)


df_runs = load_phase7_runs()
print(f"Runs found (exp=phase7_detection): {len(df_runs)}")
df_runs

Runs found (exp=phase7_detection): 13


,run_dir,model,stage,pretrained,git_hash,post_fix,status,best_val_mAP,best_val_mAP50,best_epoch,total_time_s,params_m,macs,model_size_mb,latency_ms_per_image
0,ssd_alexnet_bottleneck_fp32_phase7_detection,alexnet_bottleneck,fp32,False,baa3528eb2799b9aa61a0cbfb918a6eed1f7f758,True,complete,0.209839,0.474579,549.0,107015.732150,1.532362,4.465787e+09,5.964869,8.025433
1,ssd_alexnet_bottleneck_fp32_pretrained_phase7_...,alexnet_bottleneck,fp32,True,,None,running,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ssd_alexnet_bottleneck_int8_phase7_detection,alexnet_bottleneck,int8,False,,None,not_started,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ssd_alexnet_bottleneck_int8_pretrained_phase7_...,alexnet_bottleneck,int8,True,,None,not_started,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ssd_alexnet_bottleneck_qat_phase7_detection,alexnet_bottleneck,qat,False,4e02ab24ac1084f3578362201ece86f4e68215e3,True,complete,0.209711,0.477814,63.0,34361.207653,1.532362,4.465787e+09,6.167522,9.358138
5,ssd_alexnet_bottleneck_qat_pretrained_phase7_d...,alexnet_bottleneck,qat,True,5b578dc43677c8d542df7adb36909cb363006618,True,complete,0.202604,0.472363,47.0,52675.584173,1.532362,4.465787e+09,6.172934,8.027234
6,ssd_alexnet_fire_fp32_phase7_detection,alexnet_fire,fp32,False,f05344c9a8272b6b95d40c42a8eca25128573440,False,complete,0.004998,0.020479,25.0,9803.783799,1.108624,1.525101e+10,4.329905,16.231539
7,ssd_alexnet_fire_fp32_pretrained_phase7_detection,alexnet_fire,fp32,True,,None,running,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,ssd_alexnet_fire_qat_phase7_detection,alexnet_fire,qat,False,,None,running,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,ssd_alexnet_tv_fp32_phase7_detection,alexnet_tv,fp32,False,f05344c9a8272b6b95d40c42a8eca25128573440,False,complete,0.071419,0.165849,27.0,9486.553734,58.393696,3.744392e+09,222.811529,12.159216


### Run Inventory

Ground truth for "what has actually landed" — re-run this cell instead of `squeue` to check
PCAD job progress; a `running` row means a `.log` exists but `metrics.json` doesn't yet.

In [3]:
display_cols = ["run_dir", "model", "stage", "pretrained", "post_fix", "status",
                "best_val_mAP", "best_val_mAP50", "best_epoch"]
if df_runs.empty:
    print("[skip] No Phase 7 detection runs found on disk.")
else:
    print(df_runs[display_cols].sort_values(["model", "stage", "pretrained"]).to_string(index=False))

    n_invalid = (df_runs["post_fix"] == False).sum()  # noqa: E712 (explicit False, not falsy NaN)
    n_unknown = df_runs["post_fix"].isna().sum()
    if n_invalid:
        print(f"\n{n_invalid} run(s) are pre-anchor-fix and will be EXCLUDED from H1/H3 plots below.")
    if n_unknown:
        print(f"{n_unknown} run(s) have an undecidable git_hash (missing/unknown commit) and will "
              f"also be excluded from plots pending manual check.")

                                                run_dir              model stage  pretrained post_fix      status  best_val_mAP  best_val_mAP50  best_epoch
           ssd_alexnet_bottleneck_fp32_phase7_detection alexnet_bottleneck  fp32       False     True    complete      0.209839        0.474579       549.0
ssd_alexnet_bottleneck_fp32_pretrained_phase7_detection alexnet_bottleneck  fp32        True     None     running           NaN             NaN         NaN
           ssd_alexnet_bottleneck_int8_phase7_detection alexnet_bottleneck  int8       False     None not_started           NaN             NaN         NaN
ssd_alexnet_bottleneck_int8_pretrained_phase7_detection alexnet_bottleneck  int8        True     None not_started           NaN             NaN         NaN
            ssd_alexnet_bottleneck_qat_phase7_detection alexnet_bottleneck   qat       False     True    complete      0.209711        0.477814        63.0
 ssd_alexnet_bottleneck_qat_pretrained_phase7_detection alexnet_

## Setup Visualization Defaults

In [4]:
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"font.size": 11})

MODEL_ORDER = ["alexnet_bottleneck", "alexnet_fire", "alexnet_tv"]
MODEL_COLORS = {
    "alexnet_bottleneck": "#1baf7a",
    "alexnet_fire": "#eb6834",
    "alexnet_tv": "#2a78d6",  # reference/large-kernel backbone
}


def valid_runs(df: pd.DataFrame, stage: str, pretrained: bool | None = None) -> pd.DataFrame:
    """Post-fix, completed runs for a given stage — the only rows any hypothesis test may use."""
    out = df[(df["stage"] == stage) & (df["status"] == "complete") & (df["post_fix"] == True)]  # noqa: E712
    if pretrained is not None:
        out = out[out["pretrained"] == pretrained]
    return out

## H1 — Does small-kernel compensation transfer to detection?

Joins each backbone's Phase 3 classification top-1 (`results/results_aggregate/model_details_cross_phase.csv`)
to its Phase 7 detection mAP, normalized to `alexnet_tv` (the large-kernel reference) so the plot
directly answers "does the same relative ranking hold." Only post-fix, completed FP32 runs are
eligible.

In [5]:
ARCH_MAP = {"AlexNetBottleneck": "alexnet_bottleneck", "AlexNetFire": "alexnet_fire", "AlexNetTV": "alexnet_tv"}

model_details_path = PROJECT_ROOT / "results" / "results_aggregate" / "model_details_cross_phase.csv"
if model_details_path.exists():
    phase3_df = pd.read_csv(model_details_path)
    name_col = "model_name" if "model_name" in phase3_df.columns else "model"
    phase3_df = phase3_df[phase3_df[name_col].isin(ARCH_MAP)].copy()
    phase3_df["arch"] = phase3_df[name_col].map(ARCH_MAP)
else:
    print(f"[skip] {model_details_path} not found")
    phase3_df = pd.DataFrame()

fp32_valid = valid_runs(df_runs, stage="fp32", pretrained=False)

if phase3_df.empty or fp32_valid.empty:
    print("[skip] Need both Phase 3 classification data and >=1 valid (post-fix, complete) "
          "FP32 detection run to test H1.")
    h1_df = pd.DataFrame()
else:
    h1_df = fp32_valid.merge(
        phase3_df[["arch", "fp32_top1"]], left_on="model", right_on="arch", how="inner"
    )[["model", "fp32_top1", "best_val_mAP"]]
    print(h1_df.to_string(index=False))

    if "alexnet_tv" in h1_df["model"].values:
        ref = h1_df.set_index("model").loc["alexnet_tv"]
        h1_df["top1_norm"] = h1_df.set_index("model")["fp32_top1"].values / ref["fp32_top1"]
        h1_df["mAP_norm"] = h1_df.set_index("model")["best_val_mAP"].values / ref["best_val_mAP"]

        fig, ax = plt.subplots(figsize=(7, 4.5))
        x = np.arange(len(h1_df))
        width = 0.35
        ax.bar(x - width / 2, h1_df["top1_norm"], width, label="Classification top-1 (norm.)", color="#4a3aa7")
        ax.bar(x + width / 2, h1_df["mAP_norm"], width, label="Detection mAP (norm.)", color="#eb6834")
        ax.axhline(1.0, color="gray", linestyle="--", linewidth=1, label="alexnet_tv reference")
        ax.set_xticks(x)
        ax.set_xticklabels(h1_df["model"])
        ax.set_ylabel("Value / alexnet_tv value")
        ax.set_title("H1 — Classification vs. Detection Ranking, Normalized to alexnet_tv")
        ax.legend()
        fig.tight_layout()
        fig.savefig(FIGURES_DIR / "phase7_h1_ranking_transfer.png", dpi=150)
        plt.show()

        if len(h1_df) >= 3:
            rho = h1_df["fp32_top1"].corr(h1_df["best_val_mAP"], method="spearman")
            print(f"\nSpearman rho (top1 vs mAP): {rho:.3f} — n={len(h1_df)}, "
                  f"NOT statistically meaningful at this sample size; read as descriptive only.")
    else:
        print("[skip] alexnet_tv not among valid FP32 runs — cannot normalize.")

[skip] Need both Phase 3 classification data and >=1 valid (post-fix, complete) FP32 detection run to test H1.


## H2 — Does quantization robustness transfer to detection heads?

**Blocked as of this writing:** no `*_int8_*` run has a `metrics.json` (job 811101 crashed on
uncalibrated observers). This cell reports FP32→QAT drift as a partial proxy and prints the actual
blocker; it will run the real FP32-vs-INT8 comparison unmodified once INT8 metrics exist.

In [6]:
qat_valid = valid_runs(df_runs, stage="qat", pretrained=False)
int8_valid = valid_runs(df_runs, stage="int8", pretrained=False)

if int8_valid.empty:
    int8_runs_present = df_runs[(df_runs["stage"] == "int8") & (~df_runs["pretrained"])]
    print("[blocked] No INT8 detection metrics exist yet — H2 cannot be tested.")
    print(f"  INT8 run dirs present: {int8_runs_present['run_dir'].tolist() or 'none'}")
    print("  Known cause: job 811101 crashed with 'min tensor(inf) should be less than max "
          "tensor(-inf)' — observer calibration issue in the INT8 convert step, not yet fixed.")

if not fp32_valid.empty and not qat_valid.empty:
    h2_df = fp32_valid[["model", "best_val_mAP"]].rename(columns={"best_val_mAP": "fp32_mAP"}).merge(
        qat_valid[["model", "best_val_mAP"]].rename(columns={"best_val_mAP": "qat_mAP"}), on="model"
    )
    h2_df["qat_drift"] = h2_df["qat_mAP"] - h2_df["fp32_mAP"]
    print("\nFP32 -> QAT mAP drift (proxy only, not the FP32-vs-INT8 test H2 actually needs):")
    print(h2_df.to_string(index=False))
else:
    print("[skip] Need both valid FP32 and QAT runs for even the QAT-drift proxy.")

[blocked] No INT8 detection metrics exist yet — H2 cannot be tested.
  INT8 run dirs present: ['ssd_alexnet_bottleneck_int8_phase7_detection', 'ssd_alexnet_tv_int8_phase7_detection']
  Known cause: job 811101 crashed with 'min tensor(inf) should be less than max tensor(-inf)' — observer calibration issue in the INT8 convert step, not yet fixed.

FP32 -> QAT mAP drift (proxy only, not the FP32-vs-INT8 test H2 actually needs):
             model  fp32_mAP  qat_mAP  qat_drift
alexnet_bottleneck  0.209839 0.209711  -0.000127


## H3 — Is dense prediction more receptive-field-sensitive than classification?

For each backbone, detection mAP is expressed relative to that backbone's *own* classification
top-1 (not normalized across backbones as in H1) — this isolates whether a backbone is
disproportionately worse at localization than it is at classification, independent of its overall
strength.

In [7]:
if phase3_df.empty or fp32_valid.empty:
    print("[skip] Same data dependency as H1.")
else:
    h3_df = fp32_valid.merge(
        phase3_df[["arch", "fp32_top1"]], left_on="model", right_on="arch", how="inner"
    )[["model", "fp32_top1", "best_val_mAP"]]
    h3_df["mAP_per_top1"] = h3_df["best_val_mAP"] / (h3_df["fp32_top1"] / 100.0)
    print(h3_df.to_string(index=False))

    fig, ax = plt.subplots(figsize=(7, 4.5))
    colors = [MODEL_COLORS[m] for m in h3_df["model"]]
    ax.bar(h3_df["model"], h3_df["mAP_per_top1"], color=colors)
    ax.set_ylabel("Detection mAP / (Classification top-1 / 100)")
    ax.set_title("H3 — Detection Performance Relative to Each Backbone's Own Classification Score")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "phase7_h3_rf_sensitivity.png", dpi=150)
    plt.show()

[skip] Same data dependency as H1.


## H4 — Does the detection head dominate latency, reintroducing Winograd-irrelevant structure?

Per `ideas/PHASE7_PLAN.md` Task 9: reuse Phase 6's profiling utilities directly, no new profiling
code. `RUN_PROFILING` defaults to `False` — this is the only cell in the notebook that would build
and run a model, and it must never execute on the PCAD front-end (see `pcad_frontend_restriction`
in project memory). With it off, the cell falls back to each run's own recorded
`latency_ms_per_image` (already measured once per training run, no re-profiling needed) alongside
Phase 6's backbone-only latency numbers for comparison.

In [8]:
RUN_PROFILING = False  # set True only on a machine with a GPU available for interactive use

phase6_latency_path = PROJECT_ROOT / "results" / "phase_6_hardware_profiling_analysis" / "h3_latency_pareto.csv"
phase6_latency = pd.read_csv(phase6_latency_path) if phase6_latency_path.exists() else pd.DataFrame()

if not fp32_valid.empty:
    print("Recorded full-model (backbone + SSD head) latency, from each run's own metrics.json:")
    print(fp32_valid[["model", "latency_ms_per_image", "params_m", "macs", "model_size_mb"]].to_string(index=False))
else:
    print("[skip] No valid FP32 runs with recorded latency yet.")

if not phase6_latency.empty:
    print("\nPhase 6 backbone-only (classification) latency, for head-vs-backbone share comparison:")
    print(phase6_latency.to_string(index=False))
else:
    print(f"[skip] {phase6_latency_path} not found.")

if RUN_PROFILING:
    import torch
    from ml.det_seg_models import build_ssd_detector
    from ml.profiling import profile_model_latency

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    for arch in MODEL_ORDER:
        model = build_ssd_detector(arch).to(device).eval()
        latency = profile_model_latency(model, (1, 3, 512, 512), device)
        print(f"{arch}: {latency:.3f} ms/image (full SSD, live profile)")
else:
    print("\n[RUN_PROFILING=False] Skipping live model profiling — using recorded numbers above.")

Recorded full-model (backbone + SSD head) latency, from each run's own metrics.json:
             model  latency_ms_per_image  params_m         macs  model_size_mb
alexnet_bottleneck              8.025433  1.532362 4465786880.0       5.964869

Phase 6 backbone-only (classification) latency, for head-vs-backbone share comparison:
                            model precision  latency_ms  accuracy  efficiency  energy_per_accuracy
                       alexnet_tv      fp32    1.377280 32.887921   23.878896             3.336942
                        vgg_style      fp32    3.133579 51.814806   16.535341             2.438716
             alexnet_depthwisesep      fp32    3.251044 44.390011   13.654081             2.858440
               alexnet_bottleneck      int8    3.557293 44.539082   12.520497             3.089911
               alexnet_bottleneck      fp32    4.437371 44.622979   10.056174             3.960209
             alexnet_depthwisesep      int8    4.184361 41.469061    9.9104

## Segmentation Status (Part B — unstarted)

`create_voc_segmentation_loaders()` exists, but `build_deeplabv3_segmenter()` is a hardcoded
placeholder ignoring this project's own backbones, `SegmentationTrainer.fit()` is a one-line stub,
and `run_segmentation()` in `scripts/train_det_seg.py` prints "not yet implemented". B1–B7
(real DeepLabV3 head → QAT/INT8 → real trainer → CLI → reporting → cross-phase analysis extension
→ PCAD runs) all remain — deferred until detection (Part A) results are trusted end-to-end.

## Conclusions & Persist

In [9]:
if not df_runs.empty:
    export_path = RESULTS_DIR / "phase7_comparison.csv"
    df_runs.to_csv(export_path, index=False)
    print(f"Exported {len(df_runs)} rows -> {export_path}")
else:
    print("[skip] No runs to export.")

print("""
Limitations (n as of this writing):
- Only alexnet_bottleneck has a valid (post-fix, completed) non-pretrained FP32 run; fire/tv
  non-pretrained FP32 runs on disk are pre-anchor-fix and excluded. The _pretrained variants for
  fire/tv are still training/queued on PCAD (see Run Inventory above) — re-run this notebook once
  they land to fill in H1/H3 for all three backbones.
- H2 (quantization robustness) is blocked entirely: zero valid INT8 detection metrics exist.
- Any correlation statistic (Spearman rho) computed here is on n<=3 and has essentially no
  statistical power -- read as descriptive, not evidence.
- alexnet_tv's anchor recall (0.932) sits below the 0.95 target; accepted as a residual gap from
  its structurally coarser native feature pyramid, not chased further (ideas/PHASE7_PLAN.md
  Stage 9 log) -- alexnet_tv detection runs use --skip-anchor-check.
- Segmentation (Part B) contributes nothing to this notebook -- see status above.
""")

Exported 13 rows -> /home/rafael/Documents/dnn_study/results/phase_7_detection_segmentation_analysis/phase7_comparison.csv

Limitations (n as of this writing):
- Only alexnet_bottleneck has a valid (post-fix, completed) non-pretrained FP32 run; fire/tv
  non-pretrained FP32 runs on disk are pre-anchor-fix and excluded. The _pretrained variants for
  fire/tv are still training/queued on PCAD (see Run Inventory above) — re-run this notebook once
  they land to fill in H1/H3 for all three backbones.
- H2 (quantization robustness) is blocked entirely: zero valid INT8 detection metrics exist.
- Any correlation statistic (Spearman rho) computed here is on n<=3 and has essentially no
  statistical power -- read as descriptive, not evidence.
- alexnet_tv's anchor recall (0.932) sits below the 0.95 target; accepted as a residual gap from
  its structurally coarser native feature pyramid, not chased further (ideas/PHASE7_PLAN.md
  Stage 9 log) -- alexnet_tv detection runs use --skip-anchor-check